<a href="https://colab.research.google.com/github/Rohan46os/50M-Model/blob/main/BoomLLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import requests
import tiktoken

In [ ]:
Url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
data = requests.get(Url).text

In [ ]:
len(data)

1115394

In [5]:
enc = tiktoken.get_encoding("o200k_base")
encoded_tokens = enc.encode(data)

In [ ]:
len(encoded_tokens)

297606

In [ ]:
batch_size = 4
sequence_length = 1024
batch_sequence = []

for i in range(batch_size):
  starting_index = i * sequence_length
  ending_index = sequence_length + starting_index
  batch_sequence.append(encoded_tokens[starting_index:ending_index])
  torched_sequence = torch.tensor(batch_sequence)

In [ ]:
torched_sequence.size()

torch.Size([4, 1024])

In [ ]:
vocab_size = 199997
d_embed = 64
theta = 1000

In [ ]:
Embedding_table_ = nn.Embedding(vocab_size, d_embed)
Embedded_table = Embedding_table_(torched_sequence)

In [ ]:
Embedded_table.size()

torch.Size([4, 1024, 64])

In [ ]:
Embedded_table.size()

torch.Size([4, 1024, 64])

In [ ]:
# Rope Embedding
x = Embedded_table #shape[4, 1024, 64]
# x = [] # word embedding
position_of_tokens = torch.arange(0, sequence_length).unsqueeze(1)#shape[1,1024]
frequencies = 1 / theta ** (2 * torch.arange(0, d_embed // 2) / d_embed)
cos_frequencies = torch.cos(frequencies*position_of_tokens).unsqueeze(0)
sin_frequencies = torch.sin(frequencies*position_of_tokens).unsqueeze(0)

# x is the input embedding of d= 64
x_0 = x[:, :, 0::2]
x_1 = x[:, :, 1::2]

Rope_x0 = x_0*cos_frequencies-x_1*sin_frequencies
Rope_x1 = x_0*sin_frequencies+x_1*cos_frequencies

stacked_pair = torch.stack([Rope_x0, Rope_x1], dim=-1)
final_pos_vec = stacked_pair.flatten(-2)

In [ ]:
stacked_pair.size()
final_pos_vec.size()

torch.Size([4, 1024, 64])

Completed writing a tokenizer(used tiktoken and) and divided the input into the size[B, N] and then embedded the token into a 64d and then passed the embedded token through the Rope

In [ ]:
#MHA
batch_size = 4
sequence_length = 1024
d_embed = 64
no_of_heads = 4
noh_d = d_embed//no_of_heads

Query = nn.Linear(d_embed, d_embed)
Key = nn.Linear(d_embed, d_embed)
Value = nn.Linear(d_embed, d_embed)
out_proj = nn.Linear(d_embed, d_embed)


q  = Query(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
k = Key(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
v = Value(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)

#####Attention_Mechanism#######
attention_scores = ((q@k.transpose(-1, -2))/(noh_d)**0.5)#shape 4, 4, 1024, 1024]
####Masking(self attention)######
masking_matrix = torch.tril(torch.ones(sequence_length, sequence_length))
masked_attention_scores = attention_scores.masked_fill(masking_matrix ==0, float('-inf'))
########################################--Masked Attention--###############
sliding_window = 256
masking_matrix_1 = torch.tril(torch.ones(sequence_length, sequence_length), diagonal=0)
masking_matrix_2 = torch.tril(torch.ones(sequence_length, sequence_length), diagonal=-sliding_window)
final_mat = (masking_matrix_1-masking_matrix_2)
local_masked_attention_scores = attention_scores.masked_fill(final_mat ==0, float('-inf'))
soft_maxed_local_scores = torch.softmax(local_masked_attention_scores, dim=-1)
#############################-------#############################
soft_maxed_scores = torch.softmax(attention_scores, dim=-1)
final_attention_scores = (soft_maxed_scores@v).transpose(1, 2)#shape [4, 1024, 4, 16]
final_reshaped_score = final_attention_scores.reshape(batch_size, sequence_length, d_embed)
output_projection = out_proj(final_reshaped_score)

In [ ]:
#testing local attention
sliding_window = 2
masking_matrix_1 = torch.tril(torch.ones(10, 10), diagonal=0)
masking_matrix_2 = torch.tril(torch.ones(10, 10), diagonal=-sliding_window)
final_mat = (masking_matrix_1-masking_matrix_2)
final_mat

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.]])

In [ ]:
output_projection.size()

torch.Size([4, 1024, 64])

In [ ]:
########--------------------------Feed Forward Network-----------------------##########


feed_forward = nn.Sequential(
    nn.Linear(d_embed, 4*d_embed),
    nn.ReLU(),
    nn.Linear(4*d_embed, d_embed)
)
x_out = feed_forward(output_projection)


#####Final ouput head####  Required size(B, T, vocab_size) = 4, 1024, vocab_size
### Embeddding table size = vocab_size* 64

### 4, 1024, 199997 = (4, 1024, 64) * (199997, 64)
# 4, 1024, 64 * 64, 199997; so we can just transpose the Embedding table and multipy it withour output to get 4, 1024, 199997
#weight tieing for saving parameters


# Embedding_table_weights = Embedding_table_.weight
x_output = x_out@(Embedding_table_.weight).transpose(-1, -2) #[4, 1024, 199997]  here we used the Embedding_table_ weights as matrix to project the output in to vocab size to find probabilities

## to predict the next word we only need the last token so
logits = x_output[:, -1, :]

final_probs = torch.softmax(logits, dim=-1)


logit_selection = torch.multinomial(final_probs, num_samples=1) # We get the final token which is the index next predicted token




In [ ]:
import torch
import torch.nn as nn
import requests
import tiktoken

Url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
data = requests.get(Url).text

enc = tiktoken.get_encoding("o200k_base")
encoded_tokens = enc.encode(data)

batch_size = 4
sequence_length = 1024
batch_sequence = []

batch_size = 4
sequence_length = 1024
d_embed = 64
no_of_heads = 4
noh_d = d_embed//no_of_heads

for i in range(batch_size):
  starting_index = i * sequence_length
  ending_index = sequence_length + starting_index
  batch_sequence.append(encoded_tokens[starting_index:ending_index])
torched_sequence = torch.tensor(batch_sequence)

vocab_size = 199997
d_embed = 64
theta = 1000

Embedding_table_ = nn.Embedding(vocab_size, d_embed)
Embedded_table = Embedding_table_(torched_sequence)

Query = nn.Linear(d_embed, d_embed)
Key = nn.Linear(d_embed, d_embed)
Value = nn.Linear(d_embed, d_embed)
out_proj = nn.Linear(d_embed, d_embed)


feed_forward = nn.Sequential(
    nn.Linear(d_embed, 4*d_embed),
    nn.ReLU(),
    nn.Linear(4*d_embed, d_embed)
)

###generation of next block
idx = torched_sequence

for _ in range(100):
  # Rope Embedding
  idx_cond = idx[:, -1024:]
  x = Embedding_table_(idx_cond) #shape[4, 1024, 64]
  # x = [] # word embedding
  position_of_tokens = torch.arange(0, sequence_length).unsqueeze(1)#shape[1,1024]
  frequencies = 1 / theta ** (2 * torch.arange(0, d_embed // 2) / d_embed)
  cos_frequencies = torch.cos(frequencies*position_of_tokens).unsqueeze(0)
  sin_frequencies = torch.sin(frequencies*position_of_tokens).unsqueeze(0)

  # x is the input embedding of d= 64
  x_0 = x[:, :, 0::2]
  x_1 = x[:, :, 1::2]

  Rope_x0 = x_0*cos_frequencies-x_1*sin_frequencies
  Rope_x1 = x_0*sin_frequencies+x_1*cos_frequencies

  stacked_pair = torch.stack([Rope_x0, Rope_x1], dim=-1)
  final_pos_vec = stacked_pair.flatten(-2)

  #MHA


  q  = Query(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
  k = Key(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
  v = Value(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)

  #####Attention_Mechanism#######
  attention_scores = ((q@k.transpose(-1, -2))/(noh_d)**0.5)#shape 4, 4, 1024, 1024]
  ####Masking(self attention)######
  masking_matrix = torch.tril(torch.ones(sequence_length, sequence_length))
  masked_attention_scores = attention_scores.masked_fill(masking_matrix ==0, float('-inf'))

  soft_maxed_scores = torch.softmax(masked_attention_scores, dim=-1)
  final_attention_scores = (soft_maxed_scores@v).transpose(1, 2)#shape [4, 1024, 4, 16]
  final_reshaped_score = final_attention_scores.reshape(batch_size, sequence_length, d_embed)

  output_projection = out_proj(final_reshaped_score)

  x_out = feed_forward(output_projection)



  #####Final ouput head####  Required size(B, T, vocab_size) = 4, 1024, vocab_size
  ### Embeddding table size = vocab_size* 64

  ### 4, 1024, 199997 = (4, 1024, 64) * (199997, 64)
  # 4, 1024, 64 * 64, 199997; so we can just transpose the Embedding table and multipy it withour output to get 4, 1024, 199997
  #weight tieing for saving parameters


  # Embedding_table_weights = Embedding_table_.weight
  x_output = x_out@(Embedding_table_.weight).transpose(-1, -2) #[4, 1024, 199997]  here we used the Embedding_table_ weights as matrix to project the output in to vocab size to find probabilities

  ## to predict the next word we only need the last token so
  logits = x_output[:, -1, :]

  final_probs = torch.softmax(logits, dim=-1)


  logit_selection = torch.multinomial(final_probs, num_samples=1) # We get the final token which is the index next predicted token


  idx = torch.cat((idx, logit_selection), dim=1)

In [ ]:
for i in range(batch_size):
    print(f"\n--- Sequence {i} ---")
    print(enc.decode(idx[i].tolist()))

Properly structred it and ready to train

In [1]:
import torch
import torch.nn as nn
import requests
import tiktoken
import torch.nn.functional as F

Url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
data = requests.get(Url).text

batch_size = 4
d_embed = 64
vocab_size = 199997
sequence_length = 128
theta = 1000
noh_d = 16
no_of_heads = 4
no_layers = 4


class Transformer_block(nn.Module):
  def __init__(self):
    super().__init__()

    self.Query = nn.Linear(d_embed, d_embed)
    self.Key = nn.Linear(d_embed, d_embed)
    self.Value = nn.Linear(d_embed, d_embed)
    self.out_proj = nn.Linear(d_embed, d_embed)


    self.feed_forward = nn.Sequential(
        nn.Linear(d_embed, 4*d_embed),
        nn.ReLU(),
        nn.Linear(4*d_embed, d_embed)
    )

    self.norm1 = nn.LayerNorm(d_embed)
    self.norm2 = nn.LayerNorm(d_embed)

  def Apply_ROPE(self, x):
    T = x.shape[-2]

    #first embedded input into d_dimensoins
    #apply it only to q, k after finding key and query of input vector(x)
    position_of_tokens = torch.arange(T, device=x.device).unsqueeze(1)#shape[1,1024]
    frequencies = 1 / theta ** (2 * torch.arange(0, noh_d // 2, device=x.device) / noh_d)
    cos_frequencies = torch.cos(frequencies*position_of_tokens).unsqueeze(0)
    sin_frequencies = torch.sin(frequencies*position_of_tokens).unsqueeze(0)

    # x is the input embedding of d= 64
    x_0 = x[..., 0::2] # this takes only the last dimension of the vector
    x_1 = x[..., 1::2]

    Rope_x0 = x_0*cos_frequencies-x_1*sin_frequencies
    Rope_x1 = x_0*sin_frequencies+x_1*cos_frequencies

    stacked_pair = torch.stack([Rope_x0, Rope_x1], dim=-1)
    x = stacked_pair.flatten(-2)
    return x

  def Attention_self(self, x):
    B, T, C = x.shape
    q = self.Query(x).reshape(B, T, no_of_heads, noh_d).transpose(1,2)
    k = self.Key(x).reshape(B, T, no_of_heads, noh_d).transpose(1,2)
    v = self.Value(x).reshape(B, T, no_of_heads, noh_d).transpose(1,2)

    q = self.Apply_ROPE(q)
    k = self.Apply_ROPE(k)


    #####Attention_Mechanism#######
    attention_scores = ((q@k.transpose(-1, -2))/(noh_d )**0.5)#shape 4, 4, 1024, 1024]
    ####Masking(self attention)######
    masking_matrix = torch.tril(torch.ones(T, T, device=x.device))
    masked_attention_scores = attention_scores.masked_fill(masking_matrix ==0, float('-inf'))

    soft_maxed_scores = torch.softmax(masked_attention_scores, dim=-1)
    final_attention_scores = (soft_maxed_scores@v).transpose(1, 2)#shape [4, 1024, 4, 16]
    final_reshaped_score = final_attention_scores.reshape(batch_size, sequence_length, d_embed)

    x_attn = self.out_proj(final_reshaped_score)
    return x_attn
  # or x = x+norm1(x_attn)

  def FFN(self, x):
    x_fnn = self.feed_forward(x) # replace x_attn with x and x = x + FNN(x)
    return x_fnn

  def forward(self, x):
    x = x + self.Attention_self(self.norm1(x))
    x = x + self.FFN(self.norm2(x))
    return x








class GPT(nn.Module):
  def __init__(self, data):
    super().__init__()
    self.Embedding_table_ = nn.Embedding(vocab_size, d_embed)
    self.lm_head = nn.Linear(d_embed, vocab_size)
    self.layers = nn.ModuleList([Transformer_block() for _ in range(no_layers)])

##step1: turning words into vectors
  def encoder_block(self, data):
      enc = tiktoken.get_encoding("o200k_base")
      encoded_tokens = enc.encode(data)
      return encoded_tokens

#step2: Dividing the tokens in batchs of sequence lengths
  def batching(self, encoded_tokens):
      batch_sequence = []

      for i in range(batch_size):
          starting_index = i * sequence_length
          ending_index = starting_index + sequence_length
          batch_sequence.append(encoded_tokens[starting_index:ending_index])
      return torch.tensor(batch_sequence)  # size [B, N]

#step3:
  def forward(self, batch_sequence):
    x = self.Embedding_table_(batch_sequence)
    for layer in self.layers:
      x = layer(x)
    logits = self.lm_head(x)



    return logits










In [2]:
def get_batch(encoded_tokens):

    max_start = len(encoded_tokens) - sequence_length - 1

    starts = torch.randint(
        0,
        max_start,
        (batch_size,)
    )

    x = torch.stack([
        torch.tensor(
            encoded_tokens[i:i + sequence_length]
        )
        for i in starts
    ])

    y = torch.stack([
        torch.tensor(
            encoded_tokens[i + 1:i + sequence_length + 1]
        )
        for i in starts
    ])

    return x, y

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

model = GPT(data).to(device)

encoded_tokens = model.encoder_block(data)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

Using: cuda


In [7]:
for step in range(100):

    x, y = get_batch(encoded_tokens)

    x = x.to(device)
    y = y.to(device)

    logits = model(x)

    B, T, V = logits.shape

    loss = F.cross_entropy(
        logits.reshape(B * T, V),
        y.reshape(B * T)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 10 == 0:
        print(
            f"step {step}, loss = {loss.item():.4f}"
        )

step 0, loss = 6.7389
step 10, loss = 6.4535
step 20, loss = 6.7378
step 30, loss = 6.3931
step 40, loss = 7.1616
step 50, loss = 6.5037
step 60, loss = 6.7943
step 70, loss = 6.5026
step 80, loss = 6.5758
step 90, loss = 6.3819
